# PCOS Hormonal EDA and Invasive-Feature Usefulness

## Introduction
This notebook is performing the exploratory analysis for the cleaned infertility sidecar dataset. The table is functioning as a hormonal subset of the main clinical PCOS cohort rather than as an independent external dataset.

The notebook is using `cleaned_data/PCOS_infertility_cleaned.csv` and is focusing on the three available hormonal markers: `amh_ng_ml`, `beta_hcg_i_miu_ml`, and `beta_hcg_ii_miu_ml`. The main goal is determining how useful these invasive markers appear for interpretation and later ablation work.

## Research Positioning
This notebook is not performing standalone model training and is not treating the hormonal sidecar as a separate source population. The workflow is instead examining whether these markers are adding clinically meaningful separation beyond the stronger routine and symptom-driven signals already seen in the main clinical notebook.


## Reproducibility Setup and Path Configuration

This section is preparing the notebook environment and the figure-export directory for the hormonal EDA workflow.


In [ ]:
# Importing the libraries is supporting tabular analysis, plotting, and notebook display.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Fixing the random seed is keeping stochastic behavior stable across reruns.
np.random.seed(42)

# Configuring the plotting theme is keeping the visuals aligned with the clinical notebook style.
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

# Resolving the project root is keeping the notebook portable across launch locations.
def resolve_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for candidate in candidates:
        if (candidate / "cleaned_data").exists() and (candidate / "images").exists():
            return candidate
    return current

PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "cleaned_data" / "PCOS_infertility_cleaned.csv"
IMAGE_DIR = PROJECT_ROOT / "images" / "eda" / "hormonal"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

PCOS_LABEL_ORDER = ["PCOS Negative", "PCOS Positive"]
PCOS_LABEL_PALETTE = {
    "PCOS Negative": "#3b82f6",
    "PCOS Positive": "#e76f51",
}

def save_figure(fig: plt.Figure, slug: str) -> Path:
    output_path = IMAGE_DIR / slug
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return output_path

def grouped_numeric_summary(data: pd.DataFrame, feature: str) -> pd.DataFrame:
    return (
        data.groupby("pcos_label")[feature]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={"count": "n"})
        .round(3)
        .reset_index()
    )

def plot_numeric_by_target(
    data: pd.DataFrame,
    feature: str,
    ylabel: str,
    title: str,
    slug: str,
    log_scale: bool = False,
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(
        data=data,
        x="pcos_label",
        y=feature,
        order=PCOS_LABEL_ORDER,
        palette=PCOS_LABEL_PALETTE,
        ax=ax,
    )
    sns.stripplot(
        data=data,
        x="pcos_label",
        y=feature,
        order=PCOS_LABEL_ORDER,
        color="#264653",
        alpha=0.25,
        size=3,
        jitter=0.18,
        ax=ax,
    )
    if log_scale:
        ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    save_figure(fig, slug)
    plt.show()


## Data Loading and Derived Hormonal Views

This section is loading the cleaned hormonal sidecar, verifying the expected schema, and creating log-scale versions of the markers so that skewed distributions can be examined more clearly.


In [ ]:
# Loading the hormonal sidecar dataset is bringing the invasive-marker subset into memory for analysis.
df = pd.read_csv(DATA_PATH)

# Verifying the schema is protecting the notebook from upstream naming drift.
required_columns = [
    "pcos_y_n",
    "beta_hcg_i_miu_ml",
    "beta_hcg_ii_miu_ml",
    "amh_ng_ml",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Casting the marker columns to numeric is keeping the later plots and correlations explicit.
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Creating log-scale views is making the heavy right-skew of the hormone markers easier to inspect.
df["pcos_y_n"] = df["pcos_y_n"].astype(int)
df["pcos_label"] = df["pcos_y_n"].map({0: "PCOS Negative", 1: "PCOS Positive"})
df["log_beta_hcg_i"] = np.log10(df["beta_hcg_i_miu_ml"] + 1)
df["log_beta_hcg_ii"] = np.log10(df["beta_hcg_ii_miu_ml"] + 1)
df["log_amh"] = np.log10(df["amh_ng_ml"] + 1)

display(df.head())


## Question 1

### What is the sidecar hormonal dataset size, schema, and target balance?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the overview tables is showing the sidecar structure before the hormone-by-hormone analysis begins.
overview_q01 = pd.DataFrame(
    {
        "metric": ["row_count", "column_count", "missing_cells"],
        "value": [df.shape[0], df.shape[1], int(df.isna().sum().sum())],
    }
)
target_q01 = (
    df["pcos_label"]
    .value_counts()
    .reindex(PCOS_LABEL_ORDER)
    .rename_axis("pcos_label")
    .reset_index(name="count")
)
target_q01["percentage"] = (100 * target_q01["count"] / target_q01["count"].sum()).round(1)
schema_q01 = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values})

display(overview_q01)
display(schema_q01)
display(target_q01)


In [ ]:
# Plotting the target distribution is showing that the hormonal file mirrors the labeled portion of the main clinical cohort.
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=target_q01,
    x="pcos_label",
    y="count",
    order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 1: Hormonal Sidecar Target Distribution")
ax.set_xlabel("")
ax.set_ylabel("Participant Count")
save_figure(fig, "q01_hormonal_target_distribution.png")
plt.show()


### Insight


    The hormonal table is carrying the same 364 versus 177 class split already seen in the labeled portion of the main clinical cohort. That matching target balance is reinforcing the interpretation that this file is a sidecar subset for hormonal context, not an external validation cohort.


## Question 2

### How is AMH distributed across PCOS groups?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `amh_ng_ml` before plotting.
summary_q02 = grouped_numeric_summary(df, "amh_ng_ml")
display(summary_q02)


In [ ]:
# Plotting the grouped hormone distribution is showing how `amh_ng_ml` is separating across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="amh_ng_ml",
    ylabel="AMH (ng/mL)",
    title="Question 2: How is AMH distributed across PCOS groups?",
    slug="q02_amh_vs_pcos.png",
    log_scale=False,
)


### Insight

AMH is showing a visibly higher central tendency in the PCOS-positive cohort, which is consistent with its well-known relevance in ovarian reserve and PCOS-related ovarian morphology. This makes AMH one of the strongest invasive candidates for later ablation-style comparison.


## Question 3

### How is beta-HCG I distributed across PCOS groups?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `beta_hcg_i_miu_ml` before plotting.
summary_q03 = grouped_numeric_summary(df, "beta_hcg_i_miu_ml")
display(summary_q03)


In [ ]:
# Plotting the grouped hormone distribution is showing how `beta_hcg_i_miu_ml` is separating across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="beta_hcg_i_miu_ml",
    ylabel="Beta-hCG I (mIU/mL, log scale)",
    title="Question 3: How is beta-HCG I distributed across PCOS groups?",
    slug="q03_beta_hcg_i_vs_pcos.png",
    log_scale=True,
)


### Insight

Beta-hCG I is showing a very wide and highly skewed spread, with considerable overlap between the two groups. That heavy overlap is already suggesting that beta-hCG may be noisier and less clinically targeted for PCOS discrimination than AMH.


## Question 4

### How is beta-HCG II distributed across PCOS groups?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `beta_hcg_ii_miu_ml` before plotting.
summary_q04 = grouped_numeric_summary(df, "beta_hcg_ii_miu_ml")
display(summary_q04)


In [ ]:
# Plotting the grouped hormone distribution is showing how `beta_hcg_ii_miu_ml` is separating across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="beta_hcg_ii_miu_ml",
    ylabel="Beta-hCG II (mIU/mL, log scale)",
    title="Question 4: How is beta-HCG II distributed across PCOS groups?",
    slug="q04_beta_hcg_ii_vs_pcos.png",
    log_scale=True,
)


### Insight

Beta-hCG II is also showing substantial skew and wide overlap across the PCOS groups. This pattern is making it look more like a context variable with substantial noise than like a crisp discriminatory feature for later modeling.


## Question 5

### Which hormonal marker shows the strongest group separation by effect size?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the effect-size table is ranking which hormonal markers are separating the two groups most strongly.
effect_rows_q05 = []
for feature in ["amh_ng_ml", "beta_hcg_i_miu_ml", "beta_hcg_ii_miu_ml"]:
    negative = df.loc[df["pcos_y_n"] == 0, feature].dropna()
    positive = df.loc[df["pcos_y_n"] == 1, feature].dropna()
    pooled_std = np.sqrt(((negative.std() ** 2) + (positive.std() ** 2)) / 2)
    smd = (positive.mean() - negative.mean()) / pooled_std if pooled_std else np.nan
    effect_rows_q05.append(
        {
            "feature": feature,
            "mean_negative": round(negative.mean(), 3),
            "mean_positive": round(positive.mean(), 3),
            "median_negative": round(negative.median(), 3),
            "median_positive": round(positive.median(), 3),
            "standardized_mean_difference": round(float(smd), 3),
        }
    )
summary_q05 = pd.DataFrame(effect_rows_q05).sort_values(
    "standardized_mean_difference",
    key=lambda series: series.abs(),
    ascending=False,
)
summary_q05["abs_smd"] = summary_q05["standardized_mean_difference"].abs()
display(summary_q05)


In [ ]:
# Plotting the effect sizes is showing which invasive marker is contributing the clearest group-level separation.
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=summary_q05,
    x="standardized_mean_difference",
    y="feature",
    palette="crest",
    ax=ax,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_title("Question 5: Standardized Mean Difference by Hormonal Marker")
ax.set_xlabel("Standardized Mean Difference")
ax.set_ylabel("Marker")
save_figure(fig, "q05_hormonal_effect_sizes.png")
plt.show()


### Insight


    This ranking is helping distinguish the markers that look clinically informative from the ones that are mostly noisy. If AMH is dominating the effect-size plot while beta-hCG features remain small or unstable, that is strengthening the case for keeping AMH and de-prioritizing beta-hCG in later invasive-feature experiments.


## Question 6

### How strongly are beta-HCG I and beta-HCG II correlated?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the paired-correlation table is quantifying agreement between the two beta-hCG measurements.
summary_q06 = pd.DataFrame(
    {
        "metric": ["pearson_r", "spearman_r"],
        "value": [
            round(df[["beta_hcg_i_miu_ml", "beta_hcg_ii_miu_ml"]].corr(method="pearson").iloc[0, 1], 3),
            round(df[["beta_hcg_i_miu_ml", "beta_hcg_ii_miu_ml"]].corr(method="spearman").iloc[0, 1], 3),
        ],
    }
)
display(summary_q06)


In [ ]:
# Plotting the paired scatter is showing whether the two beta-hCG measures are largely moving together.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="beta_hcg_i_miu_ml",
    y="beta_hcg_ii_miu_ml",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.70,
    s=65,
    ax=ax,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Question 6: Beta-hCG I versus Beta-hCG II")
ax.set_xlabel("Beta-hCG I (mIU/mL, log scale)")
ax.set_ylabel("Beta-hCG II (mIU/mL, log scale)")
ax.legend(title="")
save_figure(fig, "q06_beta_hcg_pair_scatter.png")
plt.show()


### Insight


    This relationship is showing whether the two beta-hCG measurements are largely repeating the same information. If the markers are moving tightly together, later modeling would gain little by keeping both unless they are showing meaningfully different alignment with the target.


## Question 7

### How strongly is AMH associated with beta-HCG I?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the pairwise-correlation table is quantifying the relationship before plotting.
summary_q07 = pd.DataFrame(
    {
        "metric": ["pearson_r", "spearman_r"],
        "value": [
            round(df[["beta_hcg_i_miu_ml", "amh_ng_ml"]].corr(method="pearson").iloc[0, 1], 3),
            round(df[["beta_hcg_i_miu_ml", "amh_ng_ml"]].corr(method="spearman").iloc[0, 1], 3),
        ],
    }
)
display(summary_q07)


In [ ]:
# Plotting the pairwise scatter is showing whether the two markers are jointly separating the PCOS groups.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="beta_hcg_i_miu_ml",
    y="amh_ng_ml",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.70,
    s=65,
    ax=ax,
)
ax.set_xscale("log")
ax.set_title("Question 7: How strongly is AMH associated with beta-HCG I?")
ax.set_xlabel("beta_hcg_i_miu_ml")
ax.set_ylabel("amh_ng_ml")
ax.legend(title="")
save_figure(fig, "q07_amh_vs_beta_hcg_i_scatter.png")
plt.show()


### Insight

This scatter is showing whether AMH is traveling together with beta-hCG I or whether the two markers are largely independent. Weak alignment would suggest that AMH is capturing a different hormonal story from the noisier beta-hCG measurements.


## Question 8

### How strongly is AMH associated with beta-HCG II?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the pairwise-correlation table is quantifying the relationship before plotting.
summary_q08 = pd.DataFrame(
    {
        "metric": ["pearson_r", "spearman_r"],
        "value": [
            round(df[["beta_hcg_ii_miu_ml", "amh_ng_ml"]].corr(method="pearson").iloc[0, 1], 3),
            round(df[["beta_hcg_ii_miu_ml", "amh_ng_ml"]].corr(method="spearman").iloc[0, 1], 3),
        ],
    }
)
display(summary_q08)


In [ ]:
# Plotting the pairwise scatter is showing whether the two markers are jointly separating the PCOS groups.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="beta_hcg_ii_miu_ml",
    y="amh_ng_ml",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.70,
    s=65,
    ax=ax,
)
ax.set_xscale("log")
ax.set_title("Question 8: How strongly is AMH associated with beta-HCG II?")
ax.set_xlabel("beta_hcg_ii_miu_ml")
ax.set_ylabel("amh_ng_ml")
ax.legend(title="")
save_figure(fig, "q08_amh_vs_beta_hcg_ii_scatter.png")
plt.show()


### Insight

This comparison is checking whether AMH and beta-hCG II are moving together in any meaningful way. Broad overlap and weak trend would further support the idea that AMH is the more clinically targeted marker for later invasive-feature work.


## Question 9

### Does a multivariate scatter of AMH versus beta-HCG I show meaningful class separation or heavy overlap?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the quartile-style summary is preparing a compact view of the joint marker spread.
summary_q09 = (
    df.groupby("pcos_label")[["amh_ng_ml", "beta_hcg_i_miu_ml"]]
    .quantile([0.25, 0.50, 0.75])
    .round(3)
)
display(summary_q09)


In [ ]:
# Plotting the AMH versus beta-hCG I space is showing whether the classes separate or overlap in two dimensions.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="beta_hcg_i_miu_ml",
    y="amh_ng_ml",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.70,
    s=70,
    ax=ax,
)
ax.set_xscale("log")
ax.set_title("Question 9: AMH versus Beta-hCG I by PCOS Status")
ax.set_xlabel("Beta-hCG I (mIU/mL, log scale)")
ax.set_ylabel("AMH (ng/mL)")
ax.legend(title="")
save_figure(fig, "q09_amh_beta_hcg_i_multivariate_scatter.png")
plt.show()


### Insight


    This multivariate view is showing whether AMH can still separate the groups even when beta-hCG is moving unpredictably. If the groups remain layered mainly along the AMH axis rather than the beta-hCG axis, that is strengthening the interpretation that AMH is the more useful invasive marker.


## Question 10

### Are the hormone distributions heavily skewed and outlier-prone?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the skewness table is quantifying how strongly the markers are leaning to the right.
summary_q10 = pd.DataFrame(
    {
        "feature": ["amh_ng_ml", "beta_hcg_i_miu_ml", "beta_hcg_ii_miu_ml"],
        "skewness": [
            round(df["amh_ng_ml"].skew(), 3),
            round(df["beta_hcg_i_miu_ml"].skew(), 3),
            round(df["beta_hcg_ii_miu_ml"].skew(), 3),
        ],
        "q95": [
            round(df["amh_ng_ml"].quantile(0.95), 3),
            round(df["beta_hcg_i_miu_ml"].quantile(0.95), 3),
            round(df["beta_hcg_ii_miu_ml"].quantile(0.95), 3),
        ],
        "max": [
            round(df["amh_ng_ml"].max(), 3),
            round(df["beta_hcg_i_miu_ml"].max(), 3),
            round(df["beta_hcg_ii_miu_ml"].max(), 3),
        ],
    }
)
display(summary_q10)

long_q10 = df.melt(
    id_vars="pcos_label",
    value_vars=["log_amh", "log_beta_hcg_i", "log_beta_hcg_ii"],
    var_name="marker",
    value_name="log_value",
)


In [ ]:
# Plotting the log-scale hormone densities is showing the extent of skew and tail behavior across markers.
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(
    data=long_q10,
    x="marker",
    y="log_value",
    palette=["#2a9d8f", "#e9c46a", "#6d597a"],
    cut=0,
    inner="quartile",
    ax=ax,
)
ax.set_title("Question 10: Log-Scaled Hormone Distribution Shapes")
ax.set_xlabel("Marker")
ax.set_ylabel("log10(value + 1)")
save_figure(fig, "q10_hormone_skewness_violin.png")
plt.show()


### Insight


    The hormone distributions are showing strong right-skew and large upper tails, especially for the beta-hCG variables. This matters because highly skewed markers can destabilize later modeling if they are entered naively, and it also supports careful skepticism when a noisy marker appears biologically less specific.


## Question 11

### What do the hormonal correlations with pcos_y_n suggest about usefulness for later invasive-feature augmentation?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the target-correlation ranking is summarizing which hormonal markers align most strongly with PCOS status.
summary_q11 = (
    df[["amh_ng_ml", "beta_hcg_i_miu_ml", "beta_hcg_ii_miu_ml", "pcos_y_n"]]
    .corr(method="spearman")["pcos_y_n"]
    .drop("pcos_y_n")
    .sort_values(key=lambda series: series.abs(), ascending=False)
    .rename("spearman_r")
    .reset_index()
    .rename(columns={"index": "feature"})
)
summary_q11["spearman_r"] = summary_q11["spearman_r"].round(3)
display(summary_q11)


In [ ]:
# Plotting the target-correlation ranking is showing which hormonal markers deserve to survive into later ablation experiments.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=summary_q11,
    x="spearman_r",
    y="feature",
    palette="flare",
    ax=ax,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_title("Question 11: Hormonal Spearman Correlation with PCOS Status")
ax.set_xlabel("Spearman Correlation")
ax.set_ylabel("Marker")
save_figure(fig, "q11_hormonal_target_correlation.png")
plt.show()


### Insight


    This ranking is summarizing the final usefulness question for the hormonal sidecar. If AMH is clearly outranking both beta-hCG markers, the evidence is supporting AMH as the invasive marker worth preserving for later ablation studies, while beta-hCG should be treated as potentially weak or noisy unless a specific clinical rationale emerges.


## Invasive-Feature Usefulness Summary

The hormonal sidecar is behaving like a focused reference notebook rather than a standalone dataset. The main signal is appearing to come from AMH, which is showing a clearer upward shift in the PCOS-positive group than either beta-hCG measurement.

The beta-hCG markers are showing heavy skew, wide overlap, and uncertain clinical specificity for PCOS. That does not make them useless, but it does mean they should be treated cautiously and should not be assumed to improve later models simply because they are laboratory measurements.

The overall interpretation is therefore favoring AMH as the primary invasive marker to preserve for later sensitivity or ablation comparisons, while beta-hCG should remain a secondary exploratory feature unless stronger evidence appears in downstream evaluation.
